# Nicolai Parfumeur Créateur — Brand Portfolio Case Study

## Objective
This notebook applies the analytical workflow developed for the Nasomatto pilot case study to the observed Nicolai Parfumeur Créateur records in Fragrantica. It examines internal brand performance and compares the brand's records with the wider industry at three ordered accord levels: dominant accord, dyad and triad.

## Data Source
The primary source is the Databricks table `main.fragrance_analytics.fragrantica_perfumes`. It provides record identifiers, fragrance names, brand names, gender classifications, ordered main accords, user ratings and rating counts (votes).

## Analytical Framework
The analysis uses three complementary benchmarking perspectives:

1. **Brand-level benchmark:** records are compared with the overall brand portfolio and, where applicable, with records in the same gender category. Median rating and median votes define the four performance segments.
2. **Industry-wide benchmark:** records are compared with the gender-matched industry population at dominant-accord, ordered-dyad and ordered-triad levels.
3. **Exact accord-profile benchmark:** records are compared with gender-matched industry records sharing the same ordered accord profile at each level. The reference population size is retained; small groups, especially N=1, require cautious interpretation.

The four segments are Standout Profiles, Hidden Favourites, Widely Explored and Quiet Profiles. The same median-based classification rules are used throughout.

## Data Interpretation
The analysis retains Fragrantica records as represented in the source table. Multiple records may refer to different concentrations, editions, years or variations of an underlying fragrance. No comprehensive product-level reconciliation or deduplication is attempted. Counts therefore describe observed source records, not necessarily distinct fragrance compositions or a complete manufacturer catalogue.

The source table remains unchanged. All transformations are performed in separate DataFrames.

## Workflow
1. Load the source data and isolate the brand records.
2. Inspect record counts, gender distribution and repeated exact fragrance names.
3. Parse ordered accord data.
4. Calculate gender-matched industry medians for dominant accords, dyads and triads.
5. Calculate the brand's overall and gender-specific internal medians.
6. Segment records using brand, industry-wide and exact-profile benchmarks.
7. Summarize segment distributions and exact-profile reference population sizes.

The notebook follows the Nasomatto pilot methodology. Results are not pre-populated; run the cells in Databricks to calculate them.

In [0]:
# 1. Load the Fragrantica source table and the functions used throughout the analysis.
from pyspark.sql import functions as F

df_fragrantica = spark.table("main.fragrance_analytics.fragrantica_perfumes")
print("Fragrantica source records:", df_fragrantica.count())

In [0]:
# 2. Isolate Nicolai Parfumeur Créateur records and inspect the observed source-record portfolio.
# Records are retained as supplied; this cell does not merge product variations.
brand_match_values = ["nicolai parfumeur createur","nicolaï parfumeur créateur","nicolai parfumeur créateur"]

brand_portfolio = (
    df_fragrantica
    .filter(F.lower(F.trim(F.col("brand_name"))).isin(*brand_match_values))
    .select("id", "fragrance_name_clean", "brand_name", "gender",
            "rating_value", "rating_count", "main_accords")
)

brand_rated = (
    brand_portfolio
    .filter(
        F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .withColumn("gender_key", F.lower(F.trim(F.col("gender"))))
)

print("Nicolai Parfumeur Créateur source records:", brand_portfolio.count())
print("Distinct record IDs:", brand_portfolio.select("id").distinct().count())
print("Records with rating and vote data:", brand_rated.count())

# Gender distribution of the observed source records.
display(
    brand_portfolio.groupBy("gender")
    .agg(F.countDistinct("id").alias("record_count"))
    .orderBy("gender")
)

# Repeated exact names only; differently named variations are not reconciled.
display(
    brand_portfolio.groupBy("fragrance_name_clean")
    .agg(F.countDistinct("id").alias("record_count"))
    .filter(F.col("record_count") > 1)
    .orderBy(F.desc("record_count"), "fragrance_name_clean")
)

In [0]:
# 3. Parse ordered accords and prepare the brand's fragrance-level profile table.
# The first accord is dominant; the first two form the ordered dyad; the first three form the triad.
# Missing accord positions are retained in the profile table rather than dropping the source record.
accords_clean = F.trim(F.col("main_accords"))
valid_accords = (
    F.col("main_accords").isNotNull()
    & (accords_clean != "")
    & (~F.lower(accords_clean).isin("null", "none", "nan", "n/a", "[]"))
)
accord_array = F.from_json(
    F.regexp_replace(
        F.regexp_replace(F.col("main_accords"), "'", '"'),
        r"\bNone\b", "null"
    ),
    "array<string>"
)

brand_fragrance_profiles = (
    brand_portfolio
    .withColumn("accords", F.when(valid_accords, accord_array))
    .withColumn("accords", F.transform("accords", lambda x: F.lower(F.trim(x))))
    .withColumn("first_accord", F.get("accords", 0))
    .withColumn("second_accord", F.get("accords", 1))
    .withColumn("third_accord", F.get("accords", 2))
    .select("id", "fragrance_name_clean", "gender", "first_accord",
            "second_accord", "third_accord", "rating_value", "rating_count")
)

display(brand_fragrance_profiles.orderBy("gender", "fragrance_name_clean"))
print("Records with a dominant accord:", brand_fragrance_profiles.filter(F.col("first_accord").isNotNull()).count())
print("Records with a complete triad:", brand_fragrance_profiles.filter(
    F.col("first_accord").isNotNull() & F.col("second_accord").isNotNull()
    & F.col("third_accord").isNotNull()
).count())

# Rated brand records with valid gender and at least one accord for benchmarking.
brand_accords = (
    brand_rated.filter(valid_accords)
    .withColumn("accords", accord_array)
    .withColumn("accords", F.transform("accords", lambda x: F.lower(F.trim(x))))
    .withColumn("dominant_accord", F.get("accords", 0))
    .withColumn("second_accord", F.get("accords", 1))
    .withColumn("third_accord", F.get("accords", 2))
    .filter(
        F.col("gender_key").isin("men", "unisex", "women")
        & F.col("dominant_accord").isNotNull()
    )
)
print("Rated brand records with accord data:", brand_accords.count())

In [0]:
# 4. Prepare the full industry reference population and calculate median benchmarks.
# Gender-matched medians are calculated separately for dominant accords, ordered dyads and ordered triads.
# Exact-profile medians use the same ordered accord combination and gender as the brand record.
industry_accords = (
    df_fragrantica
    .filter(
        valid_accords
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
        & F.lower(F.trim(F.col("gender"))).isin("men", "unisex", "women")
    )
    .withColumn("accords", accord_array)
    .withColumn("accords", F.transform("accords", lambda x: F.lower(F.trim(x))))
    .withColumn("gender_key", F.lower(F.trim(F.col("gender"))))
    .withColumn("dominant_accord", F.get("accords", 0))
    .withColumn("second_accord", F.get("accords", 1))
    .withColumn("third_accord", F.get("accords", 2))
    .filter(F.col("dominant_accord").isNotNull())
)

def calculate_benchmarks(data, profile_columns):
    valid_data = data.filter(
        F.col("id").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
    )
    gender_benchmark = (
        valid_data.groupBy("gender_key")
        .agg(
            F.countDistinct("id").alias("industry_gender_n"),
            F.percentile_approx("rating_value", 0.5, 10000).alias("industry_median_rating"),
            F.percentile_approx("rating_count", 0.5, 10000).alias("industry_median_votes")
        )
    )
    profile_benchmark = (
        valid_data.groupBy("gender_key", *profile_columns)
        .agg(
            F.countDistinct("id").alias("profile_reference_n"),
            F.percentile_approx("rating_value", 0.5, 10000).alias("profile_median_rating"),
            F.percentile_approx("rating_count", 0.5, 10000).alias("profile_median_votes")
        )
    )
    return gender_benchmark, profile_benchmark

dominant_gender_benchmark, dominant_profile_benchmark = calculate_benchmarks(
    industry_accords.filter(F.col("dominant_accord").isNotNull()), ["dominant_accord"]
)
dyad_gender_benchmark, dyad_profile_benchmark = calculate_benchmarks(
    industry_accords.filter(
        F.col("dominant_accord").isNotNull() & F.col("second_accord").isNotNull()
    ), ["dominant_accord", "second_accord"]
)
triad_gender_benchmark, triad_profile_benchmark = calculate_benchmarks(
    industry_accords.filter(
        F.col("dominant_accord").isNotNull() & F.col("second_accord").isNotNull()
        & F.col("third_accord").isNotNull()
    ), ["dominant_accord", "second_accord", "third_accord"]
)

print("Dominant-accord industry profile groups:", dominant_profile_benchmark.count())
print("Dyad industry profile groups:", dyad_profile_benchmark.count())
print("Triad industry profile groups:", triad_profile_benchmark.count())

In [0]:
# 5. Define the shared four-segment classification and fragrance benchmarking function.
# The same median-based rules are used for brand-level, industry-wide and exact-profile comparisons.
def assign_segment(rating, votes, median_rating, median_votes):
    return (
        F.when(
            (F.col(rating) >= F.col(median_rating)) & (F.col(votes) >= F.col(median_votes)),
            "Standout Profiles"
        )
        .when(
            (F.col(rating) >= F.col(median_rating)) & (F.col(votes) < F.col(median_votes)),
            "Hidden Favourites"
        )
        .when(
            (F.col(rating) < F.col(median_rating)) & (F.col(votes) >= F.col(median_votes)),
            "Widely Explored"
        )
        .otherwise("Quiet Profiles")
    )

def build_fragrance_benchmark(brand_data, gender_benchmark, profile_benchmark,
                              profile_columns, accord_level):
    b = brand_data.alias("b")
    g = gender_benchmark.alias("g")
    p = profile_benchmark.alias("p")
    join_columns = ["gender_key"] + profile_columns
    return (
        b.join(g, "gender_key", "inner")
        .join(p, join_columns, "inner")
        .select(
            F.col("b.id").alias("id"),
            F.col("b.fragrance_name_clean").alias("fragrance"),
            F.col("b.gender_key").alias("gender"),
            F.lit(accord_level).alias("accord_level"),
            F.concat_ws(" + ", *[F.col(f"b.{column}") for column in profile_columns]).alias("accord_profile"),
            F.col("b.rating_value").alias("individual_rating"),
            F.col("b.rating_count").alias("individual_votes"),
            F.col("g.industry_gender_n"),
            F.col("g.industry_median_rating"),
            F.col("g.industry_median_votes"),
            F.col("p.profile_reference_n"),
            F.col("p.profile_median_rating"),
            F.col("p.profile_median_votes")
        )
    )

In [0]:
# 6. Calculate internal brand medians and classify records against the brand's own portfolio.
# The overall benchmark covers all rated brand records; gender-specific medians are calculated separately.
# Gender sample sizes remain visible because small groups require cautious interpretation.
brand_overall_benchmark = (
    brand_rated.agg(
        F.countDistinct("id").alias("brand_reference_n"),
        F.percentile_approx("rating_value", 0.5, 10000).alias("brand_median_rating"),
        F.percentile_approx("rating_count", 0.5, 10000).alias("brand_median_votes")
    )
)
brand_gender_benchmark = (
    brand_rated.filter(F.col("gender_key").isin("men", "unisex", "women"))
    .groupBy("gender_key")
    .agg(
        F.countDistinct("id").alias("brand_gender_n"),
        F.percentile_approx("rating_value", 0.5, 10000).alias("brand_gender_median_rating"),
        F.percentile_approx("rating_count", 0.5, 10000).alias("brand_gender_median_votes")
    )
)

brand_overall_segments = (
    brand_rated.crossJoin(brand_overall_benchmark)
    .withColumn("performance_segment", assign_segment(
        "rating_value", "rating_count", "brand_median_rating", "brand_median_votes"
    ))
)
brand_gender_segments = (
    brand_rated.filter(F.col("gender_key").isin("men", "unisex", "women"))
    .join(brand_gender_benchmark, "gender_key", "inner")
    .withColumn("gender_performance_segment", assign_segment(
        "rating_value", "rating_count",
        "brand_gender_median_rating", "brand_gender_median_votes"
    ))
)

display(brand_overall_benchmark)
display(brand_gender_benchmark)
display(
    brand_overall_segments.select(
        "fragrance_name_clean", "gender", "rating_value", "rating_count", "performance_segment"
    ).orderBy("performance_segment", F.desc("rating_count"))
)
display(
    brand_gender_segments.select(
        "fragrance_name_clean", "gender", "rating_value", "rating_count",
        "brand_gender_n", "gender_performance_segment"
    ).orderBy("gender", "gender_performance_segment")
)

In [0]:
# 7. Compare brand records with gender-wide industry medians and exact accord-profile medians.
# Results are produced for dominant accords, ordered dyads and ordered triads.
# No minimum reference-population threshold is imposed; exact-profile N remains visible.
brand_dominant_results = build_fragrance_benchmark(
    brand_accords.filter(F.col("dominant_accord").isNotNull()),
    dominant_gender_benchmark, dominant_profile_benchmark,
    ["dominant_accord"], "Dominant accord"
)
brand_dyad_results = build_fragrance_benchmark(
    brand_accords.filter(
        F.col("dominant_accord").isNotNull() & F.col("second_accord").isNotNull()
    ),
    dyad_gender_benchmark, dyad_profile_benchmark,
    ["dominant_accord", "second_accord"], "Dyad"
)
brand_triad_results = build_fragrance_benchmark(
    brand_accords.filter(
        F.col("dominant_accord").isNotNull() & F.col("second_accord").isNotNull()
        & F.col("third_accord").isNotNull()
    ),
    triad_gender_benchmark, triad_profile_benchmark,
    ["dominant_accord", "second_accord", "third_accord"], "Triad"
)

brand_fragrance_benchmark_results = (
    brand_dominant_results.unionByName(brand_dyad_results).unionByName(brand_triad_results)
)

# Industry-wide segmentation uses gender-wide industry medians.
brand_industry_segments = brand_fragrance_benchmark_results.withColumn(
    "industry_performance_segment",
    assign_segment(
        "individual_rating", "individual_votes",
        "industry_median_rating", "industry_median_votes"
    )
)

# Exact-profile segmentation uses medians for the same ordered profile and gender.
brand_exact_profile_segments = brand_industry_segments.withColumn(
    "profile_performance_segment",
    assign_segment(
        "individual_rating", "individual_votes",
        "profile_median_rating", "profile_median_votes"
    )
)

display(
    brand_exact_profile_segments.select(
        "fragrance", "gender", "accord_level", "accord_profile",
        "individual_rating", "individual_votes",
        "industry_gender_n", "industry_median_rating", "industry_median_votes",
        "industry_performance_segment", "profile_reference_n",
        "profile_median_rating", "profile_median_votes", "profile_performance_segment"
    ).orderBy("gender", "fragrance", "accord_level")
)

In [0]:
# 8. Summarize brand, industry-wide and exact-profile segment distributions.
# Counts are distinct source record IDs. Exact-profile reference sizes are retained for interpretation.
brand_overall_segment_summary = (
    brand_overall_segments.groupBy("performance_segment")
    .agg(F.countDistinct("id").alias("record_count"))
    .orderBy("performance_segment")
)
brand_gender_segment_summary = (
    brand_gender_segments.groupBy("gender_key", "gender_performance_segment")
    .agg(F.countDistinct("id").alias("record_count"))
    .orderBy("gender_key", "gender_performance_segment")
)
brand_industry_segment_summary = (
    brand_industry_segments.groupBy("gender", "accord_level", "industry_performance_segment")
    .agg(F.countDistinct("id").alias("record_count"))
    .orderBy("gender", "accord_level", "industry_performance_segment")
)
brand_exact_profile_segment_summary = (
    brand_exact_profile_segments.groupBy("gender", "accord_level", "profile_performance_segment")
    .agg(F.countDistinct("id").alias("record_count"))
    .orderBy("gender", "accord_level", "profile_performance_segment")
)
brand_profile_reference_summary = (
    brand_exact_profile_segments.groupBy("accord_level", "profile_reference_n")
    .agg(F.countDistinct("id").alias("brand_record_count"))
    .orderBy("accord_level", "profile_reference_n")
)

display(brand_overall_segment_summary)
display(brand_gender_segment_summary)
display(brand_industry_segment_summary)
display(brand_exact_profile_segment_summary)
display(brand_profile_reference_summary)